# M2-T04: Discount / Markdown History Flags

Clean per-batch flags built from `markdown_applied` and `discount_pct`, using the M1-T02 resolution (the two fields are consistent, no contradictions). These feed Track 3B directly. No aggregated score is computed.

Output columns:

| Column | Meaning |
|---|---|
| `record_id` | key to join back to the main dataset |
| `markdown_flag` | 1 if a markdown was applied |
| `discount_depth` | discount as a fraction (0 = no discount) |
| `discount_tier` | none / light (10-30%) / medium (30-50%) / deep (>50%) |
| `is_light`, `is_medium`, `is_deep` | one-hot indicators of the tier |

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'config.py').exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config import RAW_DATASET_PATH, DATA_PROCESSED, TABLES_DIR

print('Project root:', ROOT)

Project root: c:\Users\Mega Store\Desktop\NHA-5-150


## 1. Load the raw data

The raw dataset is only read, never modified.

In [2]:
df = pd.read_csv(RAW_DATASET_PATH)
assert df.shape == (100_000, 42)
assert df[['record_id', 'markdown_applied', 'discount_pct']].notna().all().all()
assert df['record_id'].is_unique
df.shape

(100000, 42)

## 2. Confirm the M1-T02 finding

Every batch with a markdown has a discount, and every batch with a discount has a markdown. The discount range is 0.10 to 0.75.

In [3]:
md_on = df['markdown_applied'] == 1
has_disc = df['discount_pct'] > 0
print(pd.crosstab(df['markdown_applied'], has_disc, rownames=['markdown_applied'], colnames=['discount_pct > 0']))
assert (md_on == has_disc).all()
print('Discount range on markdown rows:', df.loc[md_on, 'discount_pct'].min(), '-', df.loc[md_on, 'discount_pct'].max())

discount_pct > 0  False  True 
markdown_applied              
0                 69349      0
1                     0  30651
Discount range on markdown rows: 0.1 - 0.75


**Conclusion:** the fields are consistent, so the flags can be built directly from them.

## 3. Build the flags

Tier cut points (30% and 50%) follow the depth bands used in the M1 EDA report.

In [4]:
flags = pd.DataFrame({'record_id': df['record_id']})
flags['markdown_flag'] = df['markdown_applied'].astype('int8')
flags['discount_depth'] = df['discount_pct'].round(2)
flags['discount_tier'] = pd.cut(
    df['discount_pct'],
    bins=[-0.001, 0.0, 0.30, 0.50, 1.0],
    labels=['none', 'light', 'medium', 'deep'],
).astype(str)
flags['is_light'] = (flags['discount_tier'] == 'light').astype('int8')
flags['is_medium'] = (flags['discount_tier'] == 'medium').astype('int8')
flags['is_deep'] = (flags['discount_tier'] == 'deep').astype('int8')
flags.head()

,record_id,markdown_flag,discount_depth,discount_tier,is_light,is_medium,is_deep
0,1,0,0.00,none,0,0,0
1,2,0,0.00,none,0,0,0
2,3,1,0.69,deep,0,0,1
3,4,0,0.00,none,0,0,0
4,5,1,0.23,light,1,0,0


## 4. Checks

In [5]:
assert len(flags) == len(df)
assert flags.isna().sum().sum() == 0
assert set(flags['markdown_flag'].unique()) <= {0, 1}
assert ((flags['markdown_flag'] == 0) == (flags['discount_tier'] == 'none')).all()
assert ((flags['markdown_flag'] == 0) == (flags['discount_depth'] == 0)).all()
assert (flags[['is_light', 'is_medium', 'is_deep']].sum(axis=1) == flags['markdown_flag']).all()
assert flags.loc[flags['is_light'] == 1, 'discount_depth'].between(0.10, 0.30).all()
assert flags.loc[flags['is_medium'] == 1, 'discount_depth'].between(0.30, 0.50, inclusive='right').all()
assert (flags.loc[flags['is_deep'] == 1, 'discount_depth'] > 0.50).all()
assert flags['discount_depth'].between(0, 0.75).all()
print('All checks passed')

All checks passed


## 5. Tier distribution

In [6]:
summary = flags['discount_tier'].value_counts().reindex(['none', 'light', 'medium', 'deep']).rename_axis('discount_tier').to_frame('batches')
summary['share_pct'] = (100 * summary['batches'] / len(flags)).round(1)
summary

,batches,share_pct
discount_tier,,
none,69349,69.3
light,18403,18.4
medium,5805,5.8
deep,6443,6.4


**Conclusion:** 69.3% of batches have no discount, 18.4% light, 5.8% medium and 6.4% deep. The deep count (6,443) matches the M1-T02 figure for discounts above 50%.

## 6. Save

The flags go to `data/processed/` (not committed). The small distribution table goes to `reports/tables/`.

In [7]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

flags.to_csv(DATA_PROCESSED / 'M2_T04_discount_flags.csv', index=False)
summary.to_csv(TABLES_DIR / 'M2_T04_discount_tier_summary.csv')
print(flags.shape)

(100000, 7)


## Deviations from plan

None.